# 8. Deskripsi Fitur & Perhitungan Manual

Bab ini menyajikan rincian teknis, formulasi matematis, serta pembuktian perhitungan manual
untuk **dua fitur domain frekuensi (spektral)** yang diekstraksi menggunakan pustaka
[TSFEL](https://tsfel.readthedocs.io/):

| Fitur | Fungsi TSFEL | Domain | Karakteristik yang Diukur |
|---|---|:---:|---|
| **Fitur 1** | `spectral_entropy(signal, fs)` | Spectral | Tingkat kerataan/kompleksitas distribusi daya spektrum frekuensi |
| **Fitur 2** | `spectral_kurtosis(signal, fs)` | Spectral | Ketajaman atau kelandaian puncak spektrum terhadap centroidnya |

**Sinyal contoh untuk verifikasi:**

- Sinyal: $x = [2, 4, 3, 6, 5, 7, 4, 8]$
- Frekuensi sampling: $f_s = 1\text{ Hz}$ (merepresentasikan data harian, 1 observasi per hari)
- Panjang data: $N = 8$

> **Catatan metodologis:** Perhitungan langkah demi langkah pada sinyal 8 titik di bawah ini
> bertujuan memvalidasi kebenaran alur algoritma dan rumus TSFEL secara eksak.
> Pada bagian akhir, hasil verifikasi juga disandingkan dengan nilai riil yang diperoleh
> pada dataset pengamatan wilayah **Sreseh, Sampang** (M. Hendrik Purwanto).


---

## 8.1 Fitur 1: `spectral_entropy(signal, fs)`

### Konsep dan Definisi

**Spectral Entropy** mengukur ketidakteraturan (kerandoman) sebaran energi sinyal pada domain frekuensi.
Konsep ini merupakan adaptasi dari *Shannon Entropy* yang diterapkan pada spektrum daya (*power spectrum*).

- Jika sinyal berupa **gelombang murni** (daya terpusat pada satu frekuensi saja), maka distribusinya sangat teratur sehingga nilai entropinya mendekati **0**.
- Jika sinyal bersifat **acak/noise putih** (daya tersebar merata di seluruh pita frekuensi), maka entropinya bernilai tinggi mendekati **1** (setelah normalisasi).

Dalam konteks pemantauan polusi atmosfer, entropi spektral yang tinggi mengindikasikan bahwa fluktuasi konsentrasi polutan sangat bervariasi dan dipengaruhi oleh banyak komponen siklus yang acak.

### Formulasi Matematis (Sesuai Implementasi TSFEL)

1. **Pembersihan Komponen DC**:
   $$x_{\text{detrend}} = x - \bar{x}$$

2. **Transformasi Fourier Cepat (Real FFT)**:
   $$X(f_k) = \text{rfft}(x_{\text{detrend}}), \quad f_k = \frac{k \cdot f_s}{N}$$

3. **Spektrum Daya (Power Spectrum)**:
   $$P(f_k) = |X(f_k)|^2$$

4. **Distribusi Probabilitas Spektral**:
   $$p_k = \frac{P(f_k)}{\sum_j P(f_j)}$$

5. **Entropi Spektral Ternormalisasi**:
   $$H = - \frac{\sum_{k, p_k > 0} p_k \log_2(p_k)}{\log_2(K)}$$
   dengan $K$ adalah banyaknya komponen frekuensi yang memiliki daya non-nol ($p_k > 0$).


### Perhitungan Manual Langkah demi Langkah

Diketahui: $x = [2, 4, 3, 6, 5, 7, 4, 8]$, $f_s = 1$, $N = 8$.

**Langkah 1: Menghitung rata-rata dan eliminasi DC**
$$\bar{x} = \frac{2+4+3+6+5+7+4+8}{8} = \frac{39}{8} = 4{,}875$$

$$x_{\text{detrend}} = [-2{,}875,\ -0{,}875,\ -1{,}875,\ 1{,}125,\ 0{,}125,\ 2{,}125,\ -0{,}875,\ 3{,}125]$$

**Langkah 2 & 3: Magnitudo FFT dan Daya $P(f_k)$**

Frekuensi evaluasi $f_k = [0{,}0,\ 0{,}125,\ 0{,}250,\ 0{,}375,\ 0{,}500]$:

| $k$ | Frekuensi ($f_k$) | Nilai Kompleks $X(f_k)$ | Magnitudo $|X(f_k)|$ | Daya $P(f_k) = |X|^2$ |
|:---:|:---:|:---:|:---:|:---:|
| 0 | 0,000 | $0{,}000 + 0{,}000j$ | 0,0000 | 0,0000 |
| 1 | 0,125 | $-3{,}707 - 4{,}536j$ | 5,8578 | 34,3137 |
| 2 | 0,250 | $-1{,}000 - 2{,}828j$ | 3,0000 | 9,0000 |
| 3 | 0,375 | $-2{,}293 + 2{,}536j$ | 3,4185 | 11,6863 |
| 4 | 0,500 | $-11{,}000 + 0{,}000j$ | 11,0000 | 121,0000 |
| **Total** | | | | **176,0000** |

Total daya spektral $\sum P(f_k) = 0 + 34{,}3137 + 9 + 11{,}6863 + 121 = 176{,}0000$.

**Langkah 4: Normalisasi Probabilitas $p_k$**
- Komponen DC ($k=0$) memiliki daya 0 sehingga diabaikan ($K = 4$ komponen aktif tersisa).
- $p_1 = 34{,}3137 / 176 = 0{,}194964$
- $p_2 = 9{,}0000 / 176 = 0{,}051136$
- $p_3 = 11{,}6863 / 176 = 0{,}066399$
- $p_4 = 121{,}0000 / 176 = 0{,}687500$

**Langkah 5: Evaluasi Entropi**
- $-p_1 \log_2(p_1) = -0{,}194964 \times (-2{,}3587) = 0{,}459866$
- $-p_2 \log_2(p_2) = -0{,}051136 \times (-4{,}2896) = 0{,}219350$
- $-p_3 \log_2(p_3) = -0{,}066399 \times (-3{,}9127) = 0{,}259800$
- $-p_4 \log_2(p_4) = -0{,}687500 \times (-0{,}5406) = 0{,}371641$

Jumlah entropi mentah:
$$H_{\text{raw}} = 0{,}459866 + 0{,}219350 + 0{,}259800 + 0{,}371641 = 1{,}310656$$

Faktor pembagi normalisasi ($K = 4$ komponen aktif):
$$\log_2(K) = \log_2(4) = 2{,}000000$$

Hasil akhir:
$$H = \frac{1{,}310656}{2{,}000000} = \mathbf{0{,}655328}$$


In [ ]:
# Verifikasi Fitur 1 menggunakan pustaka TSFEL
import numpy as np
import tsfel.feature_extraction.features as F

signal = np.array([2, 4, 3, 6, 5, 7, 4, 8], dtype=float)
fs = 1

tsfel_entropy = F.spectral_entropy(signal, fs)
print(f"Hasil Perhitungan Manual : 0.655328")
print(f"Hasil Fungsi TSFEL       : {tsfel_entropy:.6f}")
print(f"Kesesuaian Eksak         : {np.isclose(0.655328, tsfel_entropy)}")


---

## 8.2 Fitur 2: `spectral_kurtosis(signal, fs)`

### Konsep dan Definisi

**Spectral Kurtosis** mengukur derajat keruncingan (derajat ketajaman puncak) dari bentuk spektrum frekuensi sinyal relatif terhadap distribusi normal. Fitur ini diadopsi dari pustaka *The Timbre Toolbox* (Peeters et al.):

- Nilai kurtosis yang **tinggi** menunjukkan bahwa spektrum memiliki puncak tajam dominan di sekitar frekuensi tertentu (misalnya adanya lonjakan polutan berkala yang sangat konsisten).
- Nilai kurtosis yang **rendah** menunjukkan spektrum yang datar dan tersebar merata tanpa frekuensi dominan yang menonjol.

### Formulasi Matematis (Sesuai Implementasi TSFEL)

1. **Transformasi Fourier Sinyal Asli** (tanpa eliminasi DC):
   $$f_k, |X(f_k)| = \text{calc\_fft}(x, f_s)$$

2. **Bobot Magnitudo Ternormalisasi**:
   $$w_k = \frac{|X(f_k)|}{\sum_j |X(f_j)|}$$

3. **Pusat Spektrum (Spectral Centroid)**:
   $$\mu_f = \sum_k f_k \cdot w_k$$

4. **Penyebaran Spektrum (Spectral Spread / Deviasi Standar Spektral)**:
   $$\sigma_f = \sqrt{\sum_k (f_k - \mu_f)^2 \cdot w_k}$$

5. **Spectral Kurtosis**:
   $$\text{SK} = \frac{\sum_k (f_k - \mu_f)^4 \cdot w_k}{\sigma_f^4}$$


### Perhitungan Manual Langkah demi Langkah

Diketahui sinyal asli: $x = [2, 4, 3, 6, 5, 7, 4, 8]$, $f_s = 1$, $N = 8$.

**Langkah 1: Magnitudo Spektrum Fourier (Sinyal Asli)**

| $k$ | Frekuensi ($f_k$) | Magnitudo $|X(f_k)|$ | Bobot $w_k = \frac{|X|}{\sum |X|}$ |
|:---:|:---:|:---:|:---:|
| 0 | 0,000 | 39,0000 | 0,626241 |
| 1 | 0,125 | 5,8578 | 0,094061 |
| 2 | 0,250 | 3,0000 | 0,048172 |
| 3 | 0,375 | 3,4185 | 0,054893 |
| 4 | 0,500 | 11,0000 | 0,176632 |
| **Total** | | **62,2763** | **1,000000** |

**Langkah 2: Menghitung Spectral Centroid ($\mu_f$)**
$$\mu_f = (0 \times 0{,}626241) + (0{,}125 \times 0{,}094061) + (0{,}25 \times 0{,}048172) + (0{,}375 \times 0{,}054893) + (0{,}5 \times 0{,}176632)$$
$$\mu_f = 0 + 0{,}011758 + 0{,}012043 + 0{,}020585 + 0{,}088316 = \mathbf{0{,}132702}$$

**Langkah 3: Menghitung Spectral Spread ($\sigma_f$)**
Dihitung variansi spektral $\sigma_f^2 = \sum_k (f_k - \mu_f)^2 \cdot w_k$:
- $k=0$: $(0 - 0{,}132702)^2 \times 0{,}626241 = 0{,}017610 \times 0{,}626241 = 0{,}011028$
- $k=1$: $(0{,}125 - 0{,}132702)^2 \times 0{,}094061 = 0{,}000059 \times 0{,}094061 = 0{,}000006$
- $k=2$: $(0{,}250 - 0{,}132702)^2 \times 0{,}048172 = 0{,}013759 \times 0{,}048172 = 0{,}000663$
- $k=3$: $(0{,}375 - 0{,}132702)^2 \times 0{,}054893 = 0{,}058708 \times 0{,}054893 = 0{,}003223$
- $k=4$: $(0{,}500 - 0{,}132702)^2 \times 0{,}176632 = 0{,}134908 \times 0{,}176632 = 0{,}023829$

$$\sigma_f^2 = 0{,}011028 + 0{,}000006 + 0{,}000663 + 0{,}003223 + 0{,}023829 = 0{,}038748$$
$$\sigma_f = \sqrt{0{,}038748} = 0{,}196845$$
$$\sigma_f^4 = (0{,}038748)^2 = \mathbf{0{,}001501}$$

**Langkah 4: Menghitung Momen Keempat dan Spectral Kurtosis**

Tabel evaluasi pembilang $\sum_k (f_k - \mu_f)^4 \cdot w_k$:

| $k$ | $f_k$ | $(f_k - \mu_f)$ | $(f_k - \mu_f)^4$ | $w_k$ | $(f_k - \mu_f)^4 \cdot w_k$ |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 0 | 0,000 | -0,132702 | 0,000310 | 0,626241 | 0,000194 |
| 1 | 0,125 | -0,007702 | 0,000000 | 0,094061 | 0,000000 |
| 2 | 0,250 | +0,117298 | 0,000189 | 0,048172 | 0,000009 |
| 3 | 0,375 | +0,242298 | 0,003447 | 0,054893 | 0,000189 |
| 4 | 0,500 | +0,367298 | 0,018200 | 0,176632 | 0,003215 |
| **Total** | | | | | **0,003607** |

Menghitung rasio kurtosis:
$$\text{SK} = \frac{0{,}003607}{0{,}001501} = \mathbf{2{,}402571}$$


In [ ]:
# Verifikasi Fitur 2 menggunakan pustaka TSFEL
import numpy as np
import tsfel.feature_extraction.features as F

signal = np.array([2, 4, 3, 6, 5, 7, 4, 8], dtype=float)
fs = 1

tsfel_kurtosis = F.spectral_kurtosis(signal, fs)
print(f"Hasil Perhitungan Manual : 2.402571")
print(f"Hasil Fungsi TSFEL       : {tsfel_kurtosis:.6f}")
print(f"Kesesuaian Eksak         : {np.isclose(2.402571, tsfel_kurtosis)}")


---

## 8.3 Rekapitulasi dan Perbandingan dengan Dataset Riil

Tabel berikut merangkum hasil verifikasi pada sinyal uji (8 titik sampel)
serta menampilkan **nilai aktual** yang tercatat pada dataset hasil ekstraksi sekelas
untuk lokasi pengamatan **Sreseh, Sampang** (M. Hendrik Purwanto):

| Fitur | Formulasi Pokok | Sinyal Uji (Manual) | Sinyal Uji (TSFEL) | Status | Nilai Aktual (NO2, id=33) | Nilai Aktual (CO, id=7) | Nilai Aktual (SO2, id=6) |
|---|---|:---:|:---:|:---:|:---:|:---:|:---:|
| `spectral_entropy` | $-\frac{\sum p_k \log_2 p_k}{\log_2 K}$ | **0,655328** | **0,655328** | Valid $\checkmark$ | **0,750765** | **0,792770** | **0,872564** |
| `spectral_kurtosis` | $\frac{\sum (f_k - \mu_f)^4 w_k}{\sigma_f^4}$ | **2,402571** | **2,402571** | Valid $\checkmark$ | **2,216742** | **4,919090** | **1,866887** |

### Pembahasan Hasil Aktual:
1. **Spectral Entropy** pada ketiga polutan berada di kisaran $0{,}75 - 0{,}87$. Nilai yang relatif tinggi mendekati 1 ini menandakan bahwa spektrum variasi konsentrasi polutan di wilayah Sreseh memiliki spektrum frekuensi yang tersebar luas (tidak terkonsentrasi hanya pada satu siklus musiman saja).
2. **Spectral Kurtosis** pada polutan CO mencapai $4{,}919$, jauh lebih tinggi dibanding NO2 ($2{,}217$) dan SO2 ($1{,}867$). Hal ini menunjukkan bahwa sinyal CO memiliki beberapa periode fluktuasi tajam yang menghasilkan puncak-puncak spektral yang lebih tajam dibandingkan sebarannya.


In [ ]:
# Pengecekan nilai aktual langsung dari dataset proyek
import pandas as pd

files = {
    'NO2': '../data/ekstraksi_fitur_no2.csv',
    'CO':  '../data/ekstraksi_fitur_co.csv',
    'SO2': '../data/ekstraksi_fitur_so2.csv'
}

rows = []
for pollutant, path in files.items():
    df = pd.read_csv(path)
    sub = df[df['daerah'].str.contains('Sreseh|Sampang', case=False, na=False)]
    if not sub.empty:
        r = sub.iloc[0]
        rows.append({
            'Polutan': pollutant,
            'ID': r['id'],
            'Nama': r['nama'],
            'Lokasi': r['daerah'],
            'spectral_entropy': round(r['spectral_entropy'], 6),
            'spectral_kurtosis': round(r['spectral_kurtosis'], 6)
        })

pd.DataFrame(rows)
